# 🔄 Sync the Vector DB with the `data/` folder

This notebook keeps the Chroma vector database **in sync** with the files in `data/`.
Run it whenever a document gets **added**, **edited**, or **deleted**. It figures out what actually
changed and only re-embeds *that*, instead of rebuilding everything.

It writes to the **same** database (`../chroma_db`, collection `company_docs`) that the main
tutorial/query notebook reads from. So after a run, the bot just loads the DB and sees
the latest docs. No need to re-run the tutorial's Steps 3 to 5 by hand ever again.

---

## The idea in one picture

```
                 ┌── files in data/ ──┐        ┌── what's already in Chroma ──┐
   sync  =  compare │  (the "desired") │  vs.   │        (the "actual")        │
                 └────────────────────┘        └──────────────────────────────┘
                                   │
                  ┌────────────────┼─────────────────┬──────────────────┐
              NEW file         EDITED file        DELETED file      UNCHANGED file
              → embed it       → re-embed it       → remove it        → skip (fast!)
```

### How each case gets detected (the important part)

The tutorial decides "is the DB ready?" by asking *"does the folder exist?"*. That's fine for a demo,
useless for real syncing. This tracks things **explicitly** instead:

- Every stored chunk carries **metadata**: which `source` file it came from, and a
  **`source_hash`**, an MD5 fingerprint of that file's full text.
- To sync, load the current files, fingerprint each one, and compare against the fingerprints
  already in the DB:
  - **source not in DB** → new → embed all its chunks.
  - **source in DB but hash differs** → the file was edited → delete its old chunks, embed fresh ones.
  - **source in DB, hash matches** → unchanged → do nothing (this is what makes re-runs cheap).
  - **source in DB but no longer on disk** → deleted → remove its chunks.

An edit can change how many chunks a file produces, so edited files get a **delete then
re-add** (rather than a chunk-by-chunk update). That guarantees no stale leftover chunks.

There's also a **full rebuild** switch at the bottom for changing the *embedding model* (or
chunk settings) during testing. In that case every vector must be regenerated, so it wipes and
starts clean.

## Step 1: Configuration

These match the main notebook on purpose. **`EMBED_MODEL`, `CHUNK_SIZE`, and `CHUNK_OVERLAP` must
stay identical** to what the query notebook uses. Mixing embedding models or chunk sizes in one
collection gives a broken, inconsistent index.

- `REBUILD`: leave it `False` for normal syncing. Flip to `True` only to deliberately
  throw the whole DB away and re-embed everything (e.g. when switching `EMBED_MODEL`). More on this
  in the last section.

In [1]:
from pathlib import Path

# --- Model used to embed text -> vectors (MUST match the query notebook) ---
EMBED_MODEL = "nomic-embed-text"

# --- Paths (same DB the tutorial/query notebook loads) ---
DATA_DIR = Path("..") / "data"        # documents live here
DB_DIR   = "../chroma_db"             # the persisted vector database
COLLECTION_NAME = "company_docs"      # must match the query notebook

# --- Chunking knobs (MUST match the query notebook) ---
CHUNK_SIZE    = 1000
CHUNK_OVERLAP = 150

# --- Safety switch: full wipe + re-embed everything ---
REBUILD = False

print("Config loaded.")
print("  Documents:", DATA_DIR.resolve())
print("  DB dir   :", Path(DB_DIR).resolve())
print("  Embedder :", EMBED_MODEL)
print("  REBUILD  :", REBUILD)

Config loaded.
  Documents: C:\dev\Learning\rag-qa-bot\data
  DB dir   : C:\dev\Learning\rag-qa-bot\chroma_db
  Embedder : nomic-embed-text
  REBUILD  : False


## Step 2: Load the current documents from disk

Same loader logic as the tutorial: walk `data/`, pick the right LangChain loader per file type, and
return a list of `Document` objects. The **absolute file path** gets attached as each document's
`source`, so matching against what's in the DB later stays reliable (two files with the same name in
different folders won't collide).

In [2]:
from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader, TextLoader

def load_documents(folder: Path):
    documents = []
    files = [p for p in folder.rglob("*") if p.is_file()]
    if not files:
        print(f"⚠️  No files found in {folder.resolve()}. Add some documents and re-run.")
        return documents

    for path in files:
        suffix = path.suffix.lower()
        try:
            if suffix == ".pdf":
                loader = PyPDFLoader(str(path))
            elif suffix == ".docx":
                loader = Docx2txtLoader(str(path))
            elif suffix in (".txt", ".md"):
                loader = TextLoader(str(path), autodetect_encoding=True)
            else:
                print(f"  ↷ skipping unsupported file: {path.name}")
                continue
            loaded = loader.load()
            # Normalise 'source' to the absolute path so matching is unambiguous.
            abs_src = str(path.resolve())
            for d in loaded:
                d.metadata["source"] = abs_src
            documents.extend(loaded)
            print(f"  ✅ loaded {path.name}  ({len(loaded)} document object(s))")
        except Exception as e:
            print(f"  ❌ failed on {path.name}: {e}")

    print(f"\nTotal document objects loaded: {len(documents)}")
    return documents

docs = load_documents(DATA_DIR)

C:\Users\d.aikema\AppData\Local\Temp\ipykernel_31972\2219602918.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader, TextLoader


  ✅ loaded sample_company_handbook.md  (1 document object(s))
  ✅ loaded test-driven-development-by-example.pdf  (248 document object(s))
  ✅ loaded The Art of Unit Testing.pdf  (324 document object(s))

Total document objects loaded: 573


## Step 3: Fingerprint each file, then split into chunks

Two jobs here:

1. **Fingerprint every source file.** All the text a file produced gets concatenated and hashed
   (MD5). That single `source_hash` is how "unchanged" gets told apart from "edited" later. If even
   one character in the file changes, the hash changes.

2. **Split into chunks** with the same `RecursiveCharacterTextSplitter` as the tutorial, and stamp
   each chunk with a **deterministic ID** (`<source>::<chunk-number>`) plus the `source` and
   `source_hash` metadata.

The deterministic ID matters: running this twice on the same file yields the *same* IDs, so no
accidental duplicates appear.

In [3]:
import hashlib
from collections import defaultdict
from langchain_text_splitters import RecursiveCharacterTextSplitter

def fingerprint(text: str) -> str:
    return hashlib.md5(text.encode("utf-8")).hexdigest()

# 1) Group loaded text by source file and compute a per-file hash.
text_by_source = defaultdict(list)
for d in docs:
    text_by_source[d.metadata["source"]].append(d.page_content)

source_hash = {src: fingerprint("\n".join(parts)) for src, parts in text_by_source.items()}

# 2) Split into chunks, tagging deterministic IDs + fingerprints.
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
)

all_chunks = splitter.split_documents(docs)

# Assign a stable per-source chunk index so IDs are reproducible.
counter = defaultdict(int)
chunk_ids = []
for ch in all_chunks:
    src = ch.metadata["source"]
    idx = counter[src]
    counter[src] += 1
    ch.metadata["source_hash"] = source_hash[src]
    ch.metadata["chunk_index"] = idx
    chunk_ids.append(f"{src}::{idx}")

# Handy view: desired state = {source: (hash, [chunk_ids])}
desired = defaultdict(lambda: {"hash": None, "ids": []})
for cid, ch in zip(chunk_ids, all_chunks):
    src = ch.metadata["source"]
    desired[src]["hash"] = ch.metadata["source_hash"]
    desired[src]["ids"].append(cid)

print(f"Current files on disk : {len(desired)}")
print(f"Current chunks total  : {len(all_chunks)}")

Current files on disk : 3
Current chunks total  : 1161


## Step 4: Connect to the DB and read what's *already* there

Open (or create) the Chroma collection, then pull back the metadata of everything currently
stored. From that, reconstruct the DB's view of the world: for each `source` file it knows about,
which `source_hash` got stored, and which chunk IDs belong to it.

This is the "actual" state to diff against the "desired" state from Step 3.

In [4]:
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma

embeddings = OllamaEmbeddings(model=EMBED_MODEL)

vectorstore = Chroma(
    persist_directory=DB_DIR,
    embedding_function=embeddings,
    collection_name=COLLECTION_NAME,
)

# Pull existing ids + metadata (no embeddings needed -> fast).
existing = vectorstore._collection.get(include=["metadatas"])
existing_ids  = existing["ids"]
existing_meta = existing["metadatas"]

# Reconstruct the DB's per-source state.
actual = defaultdict(lambda: {"hash": None, "ids": []})
for cid, meta in zip(existing_ids, existing_meta):
    src = meta.get("source")
    actual[src]["hash"] = meta.get("source_hash")
    actual[src]["ids"].append(cid)

print(f"DB currently holds    : {len(existing_ids)} chunks")
print(f"across source files   : {len(actual)}")

DB currently holds    : 449 chunks
across source files   : 2


## Step 5: Work out what changed (the diff)

No embedding happens here. This is pure bookkeeping. Comparing the two dictionaries sorts every
source file into one of four buckets:

| Bucket        | Condition                                              | Action later            |
|---------------|--------------------------------------------------------|-------------------------|
| **new**       | on disk, not in DB                                     | add its chunks          |
| **edited**    | in both, but `source_hash` differs                     | delete old, add new     |
| **deleted**   | in DB, no longer on disk                               | delete its chunks       |
| **unchanged** | in both, `source_hash` matches                         | skip                    |

The plan gets printed so it's easy to eyeball before anything is written.

In [5]:
disk_sources = set(desired.keys())
db_sources   = set(actual.keys())

new_sources      = disk_sources - db_sources
deleted_sources  = db_sources - disk_sources
common_sources   = disk_sources & db_sources

edited_sources    = {s for s in common_sources if desired[s]["hash"] != actual[s]["hash"]}
unchanged_sources = common_sources - edited_sources

def short(s):  # prettier printing: just the file name
    return Path(s).name if s else "<unknown>"

print("── Sync plan ──────────────────────────────")
print(f"  NEW       ({len(new_sources)}): "     + ", ".join(sorted(short(s) for s in new_sources)))
print(f"  EDITED    ({len(edited_sources)}): "  + ", ".join(sorted(short(s) for s in edited_sources)))
print(f"  DELETED   ({len(deleted_sources)}): " + ", ".join(sorted(short(s) for s in deleted_sources)))
print(f"  UNCHANGED ({len(unchanged_sources)}): "+ ", ".join(sorted(short(s) for s in unchanged_sources)))
print("───────────────────────────────────────────")

if not (new_sources or edited_sources or deleted_sources):
    print("\n✨ Nothing to do. The DB is already in sync.")

── Sync plan ──────────────────────────────
  NEW       (1): The Art of Unit Testing.pdf
  EDITED    (0): 
  DELETED   (0): 
  UNCHANGED (2): sample_company_handbook.md, test-driven-development-by-example.pdf
───────────────────────────────────────────


## Step 6: Apply the changes

Now the DB actually gets touched, in a safe order:

1. **Delete** chunks for every *deleted* and *edited* source (removing an edited file's old chunks
   first means no stale fragments survive if the new version has fewer chunks).
2. **Add** chunks for every *new* and *edited* source. `add_documents` runs them through the
   embedding model and persists the vectors.

Unchanged sources are never touched. That's the whole point, and why a re-run after changing one
file is nearly instant.

In [8]:
# 1) Deletions: edited + removed sources.
ids_to_delete = []
for s in edited_sources | deleted_sources:
    ids_to_delete.extend(actual[s]["ids"])

if ids_to_delete:
    vectorstore._collection.delete(ids=ids_to_delete)
    print(f"🗑️  Deleted {len(ids_to_delete)} old chunk(s).")
else:
    print("🗑️  Nothing to delete.")

# 2) Additions: new + edited sources.
add_sources = new_sources | edited_sources
add_docs, add_ids = [], []
for cid, ch in zip(chunk_ids, all_chunks):
    if ch.metadata["source"] in add_sources:
        add_docs.append(ch)
        add_ids.append(cid)

if add_docs:
    print(f"🧠 Embedding {len(add_docs)} chunk(s), this is the slow part…")
    vectorstore.add_documents(documents=add_docs, ids=add_ids)
    print("✅ Added.")
else:
    print("➕ Nothing to add.")

print(f"\nDB now holds {vectorstore._collection.count()} chunks.")

🗑️  Nothing to delete.
🧠 Embedding 712 chunk(s), this is the slow part…


ResponseError: Post "http://127.0.0.1:58956/tokenize": dial tcp 127.0.0.1:58956: connectex: No connection could be made because the target machine actively refused it. (status code: 400)

## Step 7: Verify the sync

A quick self-check that the DB now matches disk: same set of source files, and every file's stored
`source_hash` equals the fingerprint computed this run. A ✅ here means the query
notebook can just load the DB and it'll reflect the latest docs.

In [ ]:
check = vectorstore._collection.get(include=["metadatas"])
db_now = defaultdict(set)
for meta in check["metadatas"]:
    db_now[meta.get("source")].add(meta.get("source_hash"))

ok = True
# Same set of sources?
if set(db_now.keys()) != disk_sources:
    ok = False
    print("❌ Source set mismatch between DB and disk.")
# Correct hash for each?
for s in disk_sources:
    hashes = db_now.get(s, set())
    if hashes != {desired[s]["hash"]}:
        ok = False
        print(f"❌ Hash mismatch for {short(s)}")

print("✅ In sync. DB matches disk." if ok else "⚠️  Not fully in sync (see above).")
print(f"   Files: {len(db_now)} | Chunks: {check['ids'].__len__()}")

## Optional: Full rebuild (when the embedder changes)

Incremental sync assumes every vector in the collection was produced by the **same** embedding
model. The moment `EMBED_MODEL` changes (or `CHUNK_SIZE`/`CHUNK_OVERLAP` changes), the existing
vectors live in a *different* space and can't be mixed with new ones. The only correct fix is to
**regenerate all of them**.

That's what this cell does: it deletes the whole `chroma_db/` folder and re-embeds every file from
scratch with the current config. Set `REBUILD = True` in Step 1 first (a guard below refuses to run
otherwise, so the DB can't get nuked by accident).

Use it when:
- trying a different `EMBED_MODEL` during testing, **or**
- chunking changed, **or**
- the index seems corrupted and a clean slate is easier.

In [ ]:
import shutil

if not REBUILD:
    print("REBUILD is False, skipping. Set REBUILD = True in Step 1 to enable a full rebuild.")
else:
    # 1) Wipe the persisted DB entirely.
    db_path = Path(DB_DIR)
    if db_path.exists():
        shutil.rmtree(db_path)
        print(f"🗑️  Removed {db_path.resolve()}")

    # 2) Re-embed everything from the chunks we prepared in Step 3.
    fresh = Chroma(
        persist_directory=DB_DIR,
        embedding_function=OllamaEmbeddings(model=EMBED_MODEL),
        collection_name=COLLECTION_NAME,
    )
    if all_chunks:
        print(f"🧠 Re-embedding all {len(all_chunks)} chunk(s) with '{EMBED_MODEL}'…")
        fresh.add_documents(documents=all_chunks, ids=chunk_ids)
        print(f"✅ Rebuilt. DB now holds {fresh._collection.count()} chunks.")
    else:
        print("⚠️  No chunks to embed. Is data/ empty?")

    print("\n👉 Remember to set the SAME EMBED_MODEL in the query notebook before asking questions.")